In [19]:
# Quietly load the files
.output <- source("./Functions/lecospectR.R", echo = FALSE, verbose = FALSE)

In [20]:
get_filename <- function(
    bandwidth, 
    count, 
    is_train = TRUE, 
    base_path = "Data/v2/") {
    if (is_train) {
        train_test_string <- "train"
    } else {
        train_test_string <- "test"
    }

    return(
        paste0(
            base_path,
            train_test_string,
            "_",
            bandwidth,
            "nm_",
            count,
            ".csv"
        )
    )
}

In [21]:
seeds <- c(
    #6265,
    #4041,
    #3621,
    #942,
    #3143,
    #1764,
    1378,
    3964,
    4270,
    5542,
    1816,
    2833,
    4024,
    3031,
    6389,
    1368,
    4900,
    4075,
    6232,
    7118,
    7590,
    7928,
    2725,
    2422,
    7475,
    3857,
    1652,
    2041,
    9017,
    6447
)

In [22]:
data_raw <- read.csv(file = "Data/Ground_Validation/PFT_image_spectra/PFT_Image_SpectralLib_Clean.csv")

In [23]:
indices <- get_vegetation_indices(df = data_raw, ml_model = NULL) 


In [ ]:
bands <- resample_df(df = data_raw)
colnames(bands)

In [ ]:
num_cols_initial <- ncol(data_raw)
num_cols_bands <- ncol(bands)



In [ ]:
df <- cbind(bands, indices)

In [ ]:
colnames(df)

In [ ]:
num_per_pft <- 300
train_test_data <- create_patch_balanced_sample(
    df,
    test_count = 20,
    train_count = NULL,
    verbose = FALSE
    )

split_2_train <- create_patch_balanced_sample(
    train_test_data$remainder,
    test_count = num_per_pft,
    train_count = 1000,
    verbose = FALSE
    )

In [ ]:
exclude_vars <- c('X', "UID", "ScanNum", "sample_name", "PFT", "FncGrp1", "Site")

target_variable <- "FncGrp1"

In [ ]:
calculate_model_metrics <- function(
    model, 
    test_data, 
    test_labels, 
    seed,
    n = 4,
    model_dir = "./",
    manifest_path = "./seed_and_size.csv") {
    # create predictions (ranger)

        model_id <- uuid::UUIDgenerate()

        model_dir_full <- paste0(
                model_dir,
                model_id,
                "/"
            )

        if(!dir.exists(model_dir)){
            dir.create(model_dir)
        }

        dir.create(
            model_dir_full
        )


        model_predictions <- predict(
            model,
            test_data
        )$prediction %>% as.factor()

        confusion_matrix <- caret::confusionMatrix(
            model_predictions,
            test_labels %>% as.factor() %>% to_fg0() %>% add_forb(),
            mode = "everything"
        )

        acc <- as.list(confusion_matrix$overall)$Accuracy
        print(paste0("Model Accuracy: ", acc))

        validate_model(
            model,
            save_directory = model_dir_full
        )

        aggregated_results <- aggregate_results(model_dir_full)
        r2 <- calculate_validation_r2(aggregated_results)
        rpd <- calculate_rpd(aggregated_results)

        print(r2)

        save(model, file = paste0(model_dir_full, "model.rda"))

        print(model_dir)
        plt <- plot_by_pft(
            aggregated_results,
            save_path = paste0(model_dir_full, "aggregate.html"),
            open = FALSE,
            image_path = NULL,
            aggregation = 0
        )
        
        add_model_to_manifest(
            model_id = model_id,
            model_type = "Random Forest",
            bandwidth = 5,
            max_count = 300,
            preprocessing = paste0(
                "none"
            ),
            max_correlation = "none",
            weight = "balanced",
            hyperparam1 = n,
            # oob_error = model$prediction.error,
            accuracy = acc,
            r2 = r2,
            rpd = rpd,
            seed = seed,
            logpath = manifest_path
        )
}

In [ ]:
for(seed in seeds){
    set.seed(seed)

    # split the data based on the seed
    num_per_pft <- 300
    train_test_data <- create_patch_balanced_sample(
        df,
        test_count = 20,
        train_count = NULL,
        verbose = FALSE
        )

    split_2_train <- create_patch_balanced_sample(
        train_test_data$remainder,
        test_count = num_per_pft,
        train_count = 1000,
        verbose = FALSE
        )
    labels <- split_2_train$selection[,target_variable] %>% as.factor()
    test_labels <- train_test_data$selection[,target_variable] %>% as.factor()
    
    include_vars <- setdiff(
        colnames(split_2_train$selection),
        colnames(data_raw)
        )

    train_data <- impute_spectra(
        #clip_outliers(
            subset(
                x = split_2_train$selection,
                select = include_vars
                )
            )
        #)
    test_data <- impute_spectra(
        #clip_outliers(
            subset(
                train_test_data$selection,
                select = include_vars
                )
            )
        #)
    if (("Forb" %in% levels(labels)) && !("Forb" %in% levels(test_labels))) {
        levels(test_labels) <- c(levels(test_labels), "Forb")
    }
        
    small_model <- ranger::ranger(
        num.trees = 4,
        replace = TRUE,
        classification = TRUE,
        # alpha = a,
        case.weights = targets_to_weights(labels),
        x = train_data,
        y = labels
    )

    # reset seed
    set.seed(seed)
    big_model <- ranger::ranger(
        num.trees = 1000,
        replace = TRUE,
        classification = TRUE,
        # alpha = a,
        case.weights = targets_to_weights(labels),
        x = impute_spectra(train_data),
        y = labels
    )

    # and again for better reproducability
    set.seed(seed)
    calculate_model_metrics(
        small_model,
        train_test_data$selection[, include_vars],
        train_test_data$selection[, target_variable] %>% as.factor(),
        seed,
        n = 4,
        model_dir = "./temp/"
    )

    calculate_model_metrics(
        big_model,
        train_test_data$selection[, include_vars],
        train_test_data$selection[, target_variable] %>% as.factor(),
        seed,
        n = 1000,
        model_dir = "./temp/"
    )

    print(paste0("Iteration completed for seed ", seed))
}

ERROR: Error in `[<-.data.frame`(`*tmp*`, high_outliers, column_name, value = c(`75%` = 0.78701313324806)): missing values are not allowed in subscripted assignments of data frames
